# TorchFT + Fixed-R: healthy overhead and actual head-VM loss
Two configurations only: **TorchFT + Fixed-R OFF** and **TorchFT + Fixed-R ON**.
Each has a healthy and a head-loss observation: four cases per repetition.

Real Fashion-MNIST CNN, two CPU TorchFT DDP replicas, fixed optimizer work, minimum 60 seconds of measured training per replica. The inputs are generated by normal Ray tasks owned on the head. The two training actors, driver, Lighthouse, Redis/proxy and NFS survive elsewhere. Fixed-R protects the input tasks, not actor or optimizer state. Training is never paused to force replay. Either configuration may succeed or fail.

**Existing cluster:** set its state file; run settings, sections 1, 3 (start stopped recorded VMs), 4 and 6, then 8–10. Skip section 5 if the old node setup exists. **Fresh cluster:** set EXISTING_STATE_FILE=None; run sections 1–6, then 8–10. **Resume a prepared suite:** same settings, section 7, then 8–10. Do not run all cells on resume.

Each run/resume has a 28-minute budget plus at most 90 seconds cleanup. Data download/calibration is a separate bounded preparation phase. Failed outcomes remain recorded. No results are pre-populated; the assistant has not executed the experiment.


In [ ]:
import concurrent.futures
import json
import shlex
import subprocess
import sys
import time
import uuid
from pathlib import Path

PROJECT = "research-488322"
ZONE = "us-central1-a"
IMAGE = "ray-built-v1"
NETWORK = "default"
SUBNET = "default"
MACHINE_TYPE = "e2-standard-4"  # The current image check requires four CPUs.
DISK_GB = 100
PYTHON = "/opt/ray-env/ray-dev/bin/python"

# None creates fresh VMs. To reuse an existing cluster, enter its state.json path.
EXISTING_STATE_FILE = Path(
    "/home/tejas/Downloads/ray/ray_gcp_results/"
    "rayexp-1026-225335-53ab17/state.json"
)  # Set None only when you want a fresh five-VM cluster.
SUITE = "torchft-fixedr-head-v1"
MAX_EXPERIMENT_MINUTES = 28  # Per run/resume invocation; cleanup reserves 90 seconds.
KEEPALIVE_INTERVAL_MS = 10000  # Same in every arm and on every Ray process.
KEEPALIVE_TIMEOUT_MS = 10000
REPEATS = 1  # Four observations per repetition; 1 or 2.
OBSERVATION_TIMEOUT_S = 600  # 300..600; includes startup, training and recovery.
TORCHFT_SETTINGS = dict(
    shards_per_rank=4,          # Eight normal input-producing Ray tasks in total.
    steps_per_shard=20,         # Cache one shard per replica, then fetch the next.
    fault_fraction=0.15,        # Request head stop after both replicas pass this fraction.
    progress_every=5,           # Actual stop point also depends on GCP/SSH latency.
    collective_timeout_s=180,   # Allow input recovery during GCS replacement.
    input_timeout_s=180,
)
ML_SETTINGS = dict(
    target_training_s=90,        # Calibration prediction, not a measured result.
    minimum_training_s=60,       # Measured forward/backward/optimizer wall time, per replica.
    calibration_s=10,            # Real training, outside experimental observations.
    batch_size=128,              # 32..256
    train_examples=10000,        # First 10,000 training images.
    validation_examples=2000,   # Reserved validation subset; this pilot reports test accuracy.
    test_examples=2000,         # First 2,000 official test images.
    seed=2026,
    learning_rates=[0.001, 0.001],
    minimum_accuracy=0.65,      # Sanity check; exact healthy-reference hashes also checked.
)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "gossip_benchmarks").is_dir())
HELPERS = ROOT / "gossip_benchmarks/selective_replication"
NATIVE_HELPERS = ROOT / "gossip_benchmarks/head_vm_loss"
WORKDIR = ROOT / "ray_gcp_results"
sys.path.insert(0, str(HELPERS))
sys.path.insert(0, str(NATIVE_HELPERS))

if EXISTING_STATE_FILE is not None:
    STATE_FILE = Path(EXISTING_STATE_FILE).expanduser().resolve()
    state = json.loads(STATE_FILE.read_text())
    PROJECT, ZONE, RUN_ID = state["project"], state["zone"], state["run_id"]
    NAMES = state["names"]
else:
    if "RUN_ID" not in globals():
        RUN_ID = "rayexp-" + time.strftime("%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:6]
    NAMES = [RUN_ID + "-head"] + [RUN_ID + f"-w{i}" for i in range(1, 5)]
    STATE_FILE = WORKDIR / RUN_ID / "state.json"
LOCAL = WORKDIR / RUN_ID
LOCAL.mkdir(parents=True, exist_ok=True)
HEAD, *WORKERS = NAMES
REGION = ZONE.rsplit("-", 1)[0]
print("VMs:", *NAMES, sep="\n  ")
print("State file:", STATE_FILE)


## 1. Small infrastructure helpers
Only infrastructure cells use these gcloud wrappers. They share a 28-minute setup budget starting with the first cloud command; pauses count. Preparation and each pilot invocation have separate bounded budgets, so initial provisioning plus the full matrix can take more than 30 minutes. No experiment starts automatically.


In [ ]:
def gcloud(*args, timeout=600):
    global EXPERIMENT_DEADLINE
    if "EXPERIMENT_DEADLINE" not in globals():
        EXPERIMENT_DEADLINE = time.monotonic() + 28 * 60
    if not globals().get("CLEANUP_MODE", False):
        remaining = EXPERIMENT_DEADLINE - time.monotonic()
        if remaining <= 0:
            raise TimeoutError("Infrastructure setup budget exhausted. Inspect saved state before continuing.")
        timeout = min(timeout, remaining)
    command = ["gcloud", *map(str, args), f"--project={PROJECT}", "--quiet"]
    print(shlex.join(command), flush=True)
    result = subprocess.run(command, text=True, capture_output=True, timeout=timeout)
    if result.returncode:
        raise RuntimeError(result.stderr + "\n" + result.stdout)
    return result.stdout


def ssh(name, command):
    return gcloud("compute", "ssh", name, f"--zone={ZONE}", "--tunnel-through-iap",
                  "--ssh-flag=-oConnectTimeout=15", f"--command={command}")


def scp(source, target):
    return gcloud("compute", "scp", source, target, f"--zone={ZONE}", "--tunnel-through-iap")


def run_parallel(function, items):
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as pool:
        return list(pool.map(function, items))


## 2. Inspect current instances — read only
Do not operate this cluster while another notebook is using it. Existing VMs must match the recorded ownership labels.


In [ ]:
inventory = json.loads(gcloud("compute", "instances", "list", "--format=json"))
for vm in inventory:
    if vm.get("labels", {}).get("ray-experiment"):
        print(vm["name"], vm["status"], vm["zone"].rsplit("/", 1)[-1])


## 3. Create fresh VMs or start the recorded existing VMs
Uses the default VPC; no new network. On initial reuse, this cell can start stopped VMs. Skip this cell when resuming section 8 after its budget pause. Never clear an experiment lock to bypass the checks.


In [ ]:
from gcp.capabilities import stop_command

print("Failure command:", "gcloud", *stop_command(), "--no-graceful-shutdown")
if STATE_FILE.exists():
    print("Creation already recorded for this session. Continue with setup.", STATE_FILE)
else:
    image = json.loads(gcloud("compute", "images", "describe", IMAGE, "--format=json"))
    assert image["status"] == "READY"
    gcloud("compute", "networks", "describe", NETWORK, "--format=json")
    subnet = json.loads(gcloud("compute", "networks", "subnets", "describe", SUBNET,
                               f"--region={REGION}", "--format=json"))
    assert subnet["network"].rsplit("/", 1)[-1] == NETWORK, "Subnet belongs to a different VPC"
    # Record names before creation so partial failures can be cleaned up.
    STATE_FILE.write_text(json.dumps(dict(project=PROJECT, zone=ZONE, run_id=RUN_ID,
                                         names=NAMES, image=IMAGE, network=NETWORK, subnet=SUBNET), indent=2))
    gcloud("compute", "firewall-rules", "create", RUN_ID + "-internal",
           f"--network={NETWORK}", "--allow=tcp,udp,icmp",
           f"--source-tags={RUN_ID}", f"--target-tags={RUN_ID}")
    gcloud("compute", "firewall-rules", "create", RUN_ID + "-iap",
           f"--network={NETWORK}", "--allow=tcp:22",
           "--source-ranges=35.235.240.0/20", f"--target-tags={RUN_ID}")
    gcloud("compute", "instances", "create", *NAMES, f"--zone={ZONE}",
           f"--machine-type={MACHINE_TYPE}", f"--image={IMAGE}", f"--image-project={PROJECT}",
           f"--boot-disk-size={max(DISK_GB, int(image['diskSizeGb']))}GB",
           "--boot-disk-type=pd-balanced", "--provisioning-model=STANDARD",
           f"--subnet={SUBNET}", f"--tags={RUN_ID}", f"--labels=ray-experiment={RUN_ID}",
           "--metadata=enable-oslogin=FALSE", "--no-service-account", "--no-scopes", timeout=900)

# Initial setup only. Starts recorded, ownership-checked VMs; never deletes them.
START_STOPPED_VMS = True
if START_STOPPED_VMS:
    for name in NAMES:
        vm = json.loads(gcloud("compute", "instances", "describe", name,
                               f"--zone={ZONE}", "--format=json"))
        if vm.get("labels", {}).get("ray-experiment") != RUN_ID:
            raise RuntimeError("Foreign VM: " + name)
        if vm["status"] == "TERMINATED":
            print("Starting:", name)
            gcloud("compute", "instances", "start", name, f"--zone={ZONE}", timeout=180)
        elif vm["status"] != "RUNNING":
            raise RuntimeError("Wait for transitional VM: " + name + " " + vm["status"])


## 4. Get IPs and wait for SSH
For a resume after the last head-loss observation, skip creation, node setup and preparation. Reconnect to the saved preparation in section 7 instead.


In [ ]:
nodes = []
for name in NAMES:
    vm = json.loads(gcloud("compute", "instances", "describe", name, f"--zone={ZONE}", "--format=json"))
    assert vm.get("labels", {}).get("ray-experiment") == RUN_ID, name
    assert vm["status"] == "RUNNING", (name, vm["status"])
    nodes.append(dict(name=name, ip=vm["networkInterfaces"][0]["networkIP"], head=name == HEAD))
HEAD_IP = nodes[0]["ip"]
print(json.dumps(nodes, indent=2))


def wait_ssh(name):
    for attempt in range(30):
        try:
            ssh(name, "true")
            return
        except (RuntimeError, subprocess.TimeoutExpired):
            if attempt == 29:
                raise
            time.sleep(5)


wait_ssh(HEAD)  # Initialize gcloud's local SSH key before parallel connections.
run_parallel(wait_ssh, WORKERS)


## 5. Install the visible node helper
The image already contains Ray/PyTorch. This sets up the experiment account and node identity files. It does not create head-hosted NFS: surviving storage is prepared on w4 next.


In [ ]:
def setup_node(node):
    name = node["name"]
    ssh(name, """set -e
sudo apt-get update -qq
sudo DEBIAN_FRONTEND=noninteractive apt-get -o DPkg::Lock::Timeout=120 install -y -qq nfs-common
id raybench >/dev/null 2>&1 || sudo useradd --uid 25001 --create-home --home-dir /var/lib/raybench --shell /bin/bash raybench
test "$(id -u raybench)" = 25001
sudo mkdir -p /opt/ray-gcp /srv/ray-gcp
mountpoint -q /srv/ray-gcp || sudo chown raybench:raybench /srv/ray-gcp
""")
    path = LOCAL / (name + ".json")
    path.write_text(json.dumps(dict(node, nodes=nodes, head_ip=HEAD_IP, run_id=RUN_ID)))
    scp(path, name + ":/tmp/ray-node.json")
    scp(HELPERS / "gcp/bootstrap/node_control.py", name + ":/tmp/ray-node-control.py")
    ssh(name, """set -e
sudo install -o root -g root -m 644 /tmp/ray-node.json /etc/ray-gcp.json
sudo install -o root -g root -m 644 /tmp/ray-node-control.py /opt/ray-gcp/node_control.py
""")


run_parallel(setup_node, nodes)


## 6. Prepare dependencies, inputs and fixed work
Requires the recorded VMs running. Installs psutil==7.0.0 in the existing Ray environment; does not upgrade Ray, Torch or TorchFT. Checks TorchFT imports/API and counters before the matrix. A short single-worker CNN calibration chooses a common step count; its prediction excludes TorchFT communication and input-task overhead. Actual training duration is checked in the observations.

Keep the same ML_SETTINGS/TORCHFT_SETTINGS throughout a suite. Use a new SUITE for changed settings. No automatic controller, actor, or full-job retry is enabled.


In [ ]:
from native_torchft_pilot import NativeTorchFTPilot

cloud = NativeTorchFTPilot(STATE_FILE, WORKDIR / "head-vm-loss")
environment_file = cloud.local / "environment.json"
if environment_file.exists():
    cloud.nodes = json.loads(environment_file.read_text())["nodes"]

# New experiment setup only: never revive the failed head during failure cleanup.
if cloud.describe(cloud.head)["status"] == "TERMINATED":
    if cloud.ssh(cloud.support, "test ! -d /var/lib/raybench/ray-headloss-lock", check=False).returncode:
        raise RuntimeError("Controller lock exists; inspect cleanup before starting the head")
    for name in cloud.names:
        if name != cloud.head:
            active = cloud.ssh(name, "systemctl list-units 'ray-headloss-*' 'ray-native-*' "
                                    "--state=active,activating --no-legend").stdout
            if active.strip():
                raise RuntimeError("Active services must be inspected: " + active)
    cloud.gc("compute", "instances", "start", cloud.head, "--zone=" + cloud.zone, timeout=180)
    boot_deadline = time.monotonic() + 120
    while cloud.ssh(cloud.head, "true", check=False, timeout=20).returncode:
        if time.monotonic() > boot_deadline:
            raise TimeoutError("Original head did not become SSH-reachable")
        time.sleep(2)

cloud.prepare()
preparation = cloud.prepare_ml(SUITE, ML_SETTINGS)
print("Fixed steps per TorchFT replica:", preparation["ml"]["steps"])
print("Predicted training seconds (calibration only):", preparation["predicted_training_s"])
print("Dataset:", preparation["dataset"])
print("Prepared:", cloud.local)


## 7. Reconnect after restarting the kernel — optional
Skip if section 6 just succeeded. For a prepared-suite resume, run settings with the same EXISTING_STATE_FILE, SUITE, ML_SETTINGS, TORCHFT_SETTINGS, REPEATS and timeout; set RESUME_PREPARED=True below, then run section 8. Do not repeat preparation or calibration. If the original head is stopped by the previous case, the runner starts it for the next case.


In [ ]:
RESUME_PREPARED = False
if RESUME_PREPARED:
    from native_torchft_pilot import NativeTorchFTPilot
    from gcp.capabilities import stop_command

    cloud = NativeTorchFTPilot(STATE_FILE, WORKDIR / "head-vm-loss")
    environment = json.loads((cloud.local / "environment.json").read_text())
    cloud.nodes = environment["nodes"]
    cloud.stop_args = stop_command()
    cloud.ready = True
    print("Using saved preparation:", cloud.local)


## 8. Run or resume the two-configuration comparison
Order: OFF healthy, ON healthy, OFF head loss, ON head loss. Both configurations use identical TorchFT, input tasks, resources, placement, Redis/proxy and external driver. A second repetition reverses ON/OFF order.

The driver requests head loss after both replicas reach the configured step threshold. They keep training; no gate blocks input production or training to manufacture failure. Negative outcomes, including both configurations succeeding, are retained. On a budget pause, rerun this cell unchanged: recorded cases are skipped.


In [ ]:
summary = cloud.run_torchft(
    SUITE, ML_SETTINGS, TORCHFT_SETTINGS,
    repeats=REPEATS,
    budget_s=MAX_EXPERIMENT_MINUTES * 60,
    timeout_s=OBSERVATION_TIMEOUT_S,
    keepalive_interval_ms=KEEPALIVE_INTERVAL_MS,
    keepalive_timeout_ms=KEEPALIVE_TIMEOUT_MS,
)
print("Finished matrix:", summary.get("finished_matrix", False))
print("Pause reason:", summary.get("paused_reason"))
print("Cleanup errors:", summary.get("cleanup_errors", []))


## 9. Inspect saved progress — no cloud calls
Failures before injection are setup failures, not evidence of failed recovery.


In [ ]:
REPORT = cloud.local / SUITE / "native-report.json"
report = json.loads(REPORT.read_text())
print("Recorded:", len(report["samples"]), "/", report["planned_observations"])
for row in report["samples"]:
    result = row.get("result", {})
    print("\n", row["label"], row["status"], "correct:", result.get("correct"),
          "job seconds:", result.get("elapsed_s"))
    print("  Replayed input tasks:", result.get("reexecutions"))
    for replica in result.get("replicas", []):
        print("  Replica", replica["rank"],
              "training seconds:", round(replica["compute_s"], 2),
              "steps:", replica["metrics"]["steps"],
              "test accuracy:", replica["metrics"]["test_accuracy"],
              "input wait:", round(replica["input_wait_s"], 2))
    if row["fault"]:
        print("  Valid head-loss observation:", row.get("valid_head_loss_observation"))
        print("  Correct recovery:", row.get("recovered_correctly"))
        print("  Progress around failure:", row.get("training_at_failure"))
    if result.get("error") or row.get("error"):
        print("  Error:", result.get("error") or row.get("error"))


## Plot saved results — local files only
Two configurations, no vertical bars or bottom annotations. Healthy job time includes actual startup and input work; there is no supervisor gate to subtract. The right panel distinguishes verified failure from an invalid/missing observation. A single repetition cannot establish a small speedup. This does not compare against an application retry controller.


In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt

RESULTS_DIR = None  # Or Path("/absolute/path/to/torchft-fixedr-head-v1")
PLOT_REPETITION = 0
if RESULTS_DIR is None:
    RESULTS_DIR = cloud.local / SUITE
report = json.loads((Path(RESULTS_DIR) / "native-report.json").read_text())
rows = {r["label"]: r for r in report["samples"]}
healthy = [rows.get(f"{mode}-healthy-r{PLOT_REPETITION}", {}) for mode in ("off", "on")]
loss = [rows.get(f"{mode}-loss-r{PLOT_REPETITION}", {}) for mode in ("off", "on")]
valid = all(r.get("result", {}).get("correct") and
            r.get("result", {}).get("valid_overhead_sample") for r in healthy)
overhead = None
if valid:
    a, b = [r["result"]["elapsed_s"] for r in healthy]
    overhead = 100 * (b/a - 1)
    print(f"Fixed-R healthy job-time overhead: {overhead:+.2f}% ({a:.2f}s -> {b:.2f}s)")
    print("Single paired repetition; a negative change is not proof of a speedup.")
else:
    print("No valid healthy pair yet; check failures and the 60-second training floor.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout="constrained")
labels = ["TorchFT + Fixed-R OFF", "TorchFT + Fixed-R ON"]
for y, row, color in zip([1, 0], healthy, ["#d97706", "#15803d"]):
    result = row.get("result", {})
    if result.get("correct") and result.get("valid_overhead_sample"):
        seconds = result["elapsed_s"]
        axes[0].scatter(seconds, y, s=130, color=color)
        axes[0].annotate(f"{seconds:.2f} s", (seconds, y), xytext=(10, 0),
                         textcoords="offset points", va="center")
    else:
        axes[0].text(.05, y, "No valid timing", transform=axes[0].get_yaxis_transform(), va="center")
limit = max([r.get("result", {}).get("elapsed_s", 0) for r in healthy] + [1])
axes[0].set_xlim(0, limit*1.35)
axes[0].set_title("Healthy job time" + (f"\nFixed-R overhead: {overhead:+.2f}%" if overhead is not None else ""), pad=18)
axes[0].set_xlabel("Seconds", labelpad=10)
axes[0].xaxis.set_label_position("top")
for y, row in zip([1, 0], loss):
    if row.get("valid_head_loss_observation") and "result" in row:
        ok = bool(row.get("recovered_correctly"))
        axes[1].scatter(int(ok), y, s=160, marker="o" if ok else "X",
                        color="#15803d" if ok else "#dc2626")
    else:
        axes[1].text(.05, y, "Not run / invalid injection", transform=axes[1].get_yaxis_transform(), va="center")
axes[1].set_xticks([0, 1], ["Failed", "Correct completion"])
axes[1].set_xlim(-.4, 1.6)
axes[1].set_title("Actual head-VM loss\nSame surviving infrastructure", pad=18)
for ax in axes:
    ax.set_yticks([1, 0], labels)
    ax.set_ylim(-.6, 1.6)
    ax.xaxis.tick_top()
    ax.grid(axis="x", alpha=.2)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(length=0)
plt.show()


## 10. Collect evidence before deleting VMs
Copies native log tails, task attempt evidence and Redis state from w4. Keep the local native-report.json as well. This cell does not rerun workloads. If collection fails, keep w4 and its disk until you have the evidence.


In [ ]:
print("Local results:", cloud.local / SUITE)
DOWNLOAD_EVIDENCE = True
if DOWNLOAD_EVIDENCE:
    evidence_dir = cloud.download(SUITE)
    print("Upload:", cloud.local / SUITE / "native-report.json")
    print("Upload:", evidence_dir / "evidence.tar.gz")
print("Calibration metadata:", cloud.local / ("ml-preparation-" + SUITE + ".json"))


## Diagnose an IAP/SSH failure — optional
After a connection error, restart the kernel and run the settings cell with EXISTING_STATE_FILE pointing to this cluster's saved state.json. Set DIAGNOSE_CONNECTIONS=True below. This cell checks VM identity/status, SSH reachability, support services and the suite lock; it changes no cloud state.

An IAP 4003 / port 22 error is a controller transport failure, not evidence of Ray recovery failure. The native runner now retries that specific connection-creation error up to three times within its original command/session budget. Other errors and timeouts are not blindly retried. Events are saved in native-transport.jsonl and each observation's transport_events.

Keep the old suite/report. If surviving VMs are RUNNING, SSH is reachable, cleanup succeeded, and no suite lock remains, set SUITE="fashion-head-v1", run section 6 to upload the updated helpers, then section 8. Do not use section 7 to bypass the helper update. If cleanup errors or a lock remain, inspect them before proceeding; this notebook does not automatically remove a lock. You do not need to recreate a healthy cluster solely because a connection failed.


In [ ]:
# Run independently after the settings cell. Does not start/stop VMs or Ray.
DIAGNOSE_CONNECTIONS = False
PREVIOUS_SUITE = "torchft-fixedr-head-v1"

if DIAGNOSE_CONNECTIONS:
    from native_pilot import NativePilot

    diagnostic_cloud = NativePilot(STATE_FILE, WORKDIR / "head-vm-loss")
    environment_path = diagnostic_cloud.local / "environment.json"
    if environment_path.exists():
        diagnostic_cloud.nodes = json.loads(environment_path.read_text())["nodes"]
    previous_report = diagnostic_cloud.local / PREVIOUS_SUITE / "native-report.json"
    if previous_report.exists():
        previous = json.loads(previous_report.read_text())
        print("Previous cleanup errors:", previous.get("cleanup_errors", "not recorded"))
        for row in previous.get("samples", []):
            print(row["label"], row["status"])
    for name in diagnostic_cloud.names:
        try:
            vm = diagnostic_cloud.describe(name)
            print(name, vm["status"])
            if vm["status"] == "RUNNING":
                probe = diagnostic_cloud.ssh(name, "true", check=False, timeout=30)
                print("  SSH:", "reachable" if probe.returncode == 0 else probe.stderr[-2000:])
            if name == diagnostic_cloud.support and vm["status"] == "RUNNING":
                probe = diagnostic_cloud.ssh(name, """uptime
systemctl is-active ssh
systemctl list-units 'ray-headloss-*' 'ray-native-*' --state=active,activating --no-legend
if test -d /var/lib/raybench/ray-headloss-lock; then
    printf 'Suite lock is present; inspect cleanup before proceeding.\\n'
else
    printf 'No suite lock.\\n'
fi
""", check=False, timeout=30)
                print(probe.stdout, probe.stderr)
        except Exception as exc:
            print(name, str(exc))


## 11. Optional VM stop
Stops only the five recorded, ownership-checked VMs. Disks/evidence remain allocated; no deletion. Download wanted results first. Do not stop a cluster with a live controller or unresolved cleanup errors.


In [ ]:
STOP_VMS = False
if STOP_VMS:
    if cloud.ssh(cloud.support, "test ! -d /var/lib/raybench/ray-headloss-lock", check=False).returncode:
        raise RuntimeError("Suite lock exists; inspect the controller before stopping VMs")
    for name in cloud.names:
        vm = cloud.describe(name)
        if vm["status"] == "RUNNING":
            cloud.gc("compute", "instances", "stop", name, "--zone=" + cloud.zone, timeout=300)
else:
    print("VM stop skipped. Running VMs and persistent disks remain billable.")
